# Projeto Final MBA em Engenharia de Dados e Analytics
## Modelagem histórica de risco de voos

Este notebook prepara uma **Gold/SPEC de risco futuro**.

A granularidade final é:

`data_voo × municipio_origem × municipio_destino × nome_empresa`

São produzidos:

- `score_cancelamento`: probabilidade estimada de cancelamento, entre 0 e 1;
- `score_atraso`: probabilidade estimada de atraso, entre 0 e 1;
- métricas históricas de cancelamento e atraso para explicabilidade;
- estatísticas históricas de tarifas: média, mediana, desvio padrão, Q1, Q3 e IQR;
- `preco_estimado_modelo`: referência de preço via regressão temporal;
- `score_volatilidade_preco`: risco histórico de volatilidade de preço, entre 0 e 1;
- `score_risco_operacional`: média entre cancelamento e atraso;
- `score_risco_base`: média entre cancelamento, atraso e volatilidade de preço.

> O score que compara **preço atual** com o histórico será calculado posteriormente, quando a SerpApi for integrada. Nesta etapa, a Gold já materializa todas as referências necessárias para esse cálculo.


In [ ]:
# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

import os
import gc
import math
import glob
import joblib
import numpy as np
import pandas as pd
import polars as pl

from datetime import date, timedelta

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(50)

RANDOM_STATE = 42
BATCH_SIZE = 200_000


In [ ]:
# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

# Colab Drive desativado — dados locais em data/SoR · SoT · Spec
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
# ============================================================
# 3. CONFIGURAÇÕES DO PROJETO
# ============================================================
# CONFIGURAÇÕES DO PROJETO (monorepo local)
# SoR ≈ Bronze · SoT ≈ Silver · Spec ≈ Gold
# ============================================================
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "data" / "SoR").exists():
    _data = _here / "data"
elif (_here.parent / "data" / "SoR").exists():
    _data = _here.parent / "data"
else:
    _data = _here / "data"

pasta_projeto = str(_data)
pasta_sor = str(_data / "SoR")
pasta_sot = str(_data / "SoT")
pasta_spec = str(_data / "Spec")
pasta_modelos = fr"{pasta_spec}/spec_modelos"

os.makedirs(pasta_spec, exist_ok=True)
os.makedirs(pasta_modelos, exist_ok=True)

# Ajuste somente se o nome da pasta de tarifas estiver diferente no Drive.
PATH_VOOS = fr"{pasta_sot}/SoT_historico_voos/*.parquet"
PATH_TARIFAS = fr"{pasta_sot}/SoT_tarifas/*.parquet"

hoje = date.today()
hoje_str = hoje.strftime("%Y%m%d")

DATA_INICIAL_SIMULACAO = hoje + timedelta(days=1)
DATA_FINAL_SIMULACAO = date(2027, 12, 31)

# Cortes temporais utilizados nos modelos operacionais
DATA_INICIO_VALIDACAO = date(2025, 1, 1)
DATA_INICIO_TESTE = date(2026, 1, 1)

print("Data de processamento:", hoje)
print("Simulações:", DATA_INICIAL_SIMULACAO, "até", DATA_FINAL_SIMULACAO)


## 4. Funções auxiliares

As funções abaixo centralizam:

- avaliação dos modelos;
- scoring em lotes, evitando excesso de memória;
- criação das estatísticas ponderadas de tarifa;
- materialização das métricas.


In [ ]:
# ============================================================
# 4.1 MÉTRICAS DE CLASSIFICAÇÃO
# ============================================================

def avaliar_classificador(modelo, X, y, nome_modelo, split):
    prob = modelo.predict_proba(X)[:, 1]

    metricas = {
        "modelo": nome_modelo,
        "split": split,
        "prevalencia": float(np.mean(y)),
        "roc_auc": float(roc_auc_score(y, prob)),
        "pr_auc": float(average_precision_score(y, prob)),
        "brier_score": float(brier_score_loss(y, prob)),
        "log_loss": float(log_loss(y, prob)),
    }

    return metricas, prob


# ============================================================
# 4.2 MÉTRICAS DE REGRESSÃO
# ============================================================

def avaliar_regressor(y_true, y_pred, nome_modelo, split):
    y_pred = np.maximum(np.asarray(y_pred), 0)

    return {
        "modelo": nome_modelo,
        "split": split,
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


# ============================================================
# 4.3 PREDIÇÃO EM LOTES
# ============================================================

def prever_probabilidade_em_lotes(modelo, df_polars, features, batch_size=BATCH_SIZE):
    probabilidades = []

    for inicio in range(0, df_polars.height, batch_size):
        lote = (
            df_polars
            .slice(inicio, batch_size)
            .select(features)
            .to_pandas()
        )

        probabilidades.append(
            modelo.predict_proba(lote)[:, 1]
        )

    return np.concatenate(probabilidades)


def prever_regressao_em_lotes(modelo, df_polars, features, batch_size=BATCH_SIZE):
    previsoes = []

    for inicio in range(0, df_polars.height, batch_size):
        lote = (
            df_polars
            .slice(inicio, batch_size)
            .select(features)
            .to_pandas()
        )

        pred_log = modelo.predict(lote)
        previsoes.append(np.maximum(np.expm1(pred_log), 0))

    return np.concatenate(previsoes)


In [ ]:
# ============================================================
# 4.4 ESTATÍSTICAS PONDERADAS DE PREÇO
# ============================================================

def estatisticas_preco_ponderadas(df, chaves, prefixo="preco_hist"):
    """
    Calcula estatísticas usando 'assentos' como peso.

    A ANAC informa tarifa e quantidade de assentos associados à tarifa.
    Por isso, média, desvio e quantis devem considerar o peso de assentos.
    """

    base = (
        df
        .filter(
            (pl.col("tarifa").is_not_null()) &
            (pl.col("assentos").is_not_null()) &
            (pl.col("tarifa") > 0) &
            (pl.col("assentos") > 0)
        )
        .select(chaves + ["tarifa", "assentos"])
        .with_columns([
            (pl.col("tarifa") * pl.col("assentos")).alias("_wx"),
            ((pl.col("tarifa") ** 2) * pl.col("assentos")).alias("_wx2"),
        ])
    )

    momentos = (
        base
        .group_by(chaves)
        .agg([
            pl.col("assentos").sum().alias(f"{prefixo}_assentos"),
            pl.len().alias(f"{prefixo}_qtd_registros"),
            pl.col("_wx").sum().alias("_sum_wx"),
            pl.col("_wx2").sum().alias("_sum_wx2"),
            pl.col("tarifa").min().alias(f"{prefixo}_min"),
            pl.col("tarifa").max().alias(f"{prefixo}_max"),
        ])
        .with_columns(
            (pl.col("_sum_wx") / pl.col(f"{prefixo}_assentos"))
            .alias(f"{prefixo}_media")
        )
        .with_columns(
            (
                (
                    pl.col("_sum_wx2") / pl.col(f"{prefixo}_assentos")
                    - pl.col(f"{prefixo}_media") ** 2
                )
                .clip(lower_bound=0)
                .sqrt()
            ).alias(f"{prefixo}_desvio")
        )
        .drop(["_sum_wx", "_sum_wx2"])
    )

    ordenado = (
        base
        .select(chaves + ["tarifa", "assentos"])
        .sort(chaves + ["tarifa"])
        .with_columns([
            pl.col("assentos").cum_sum().over(chaves).alias("_cum_assentos"),
            pl.col("assentos").sum().over(chaves).alias("_total_assentos"),
        ])
        .with_columns(
            (pl.col("_cum_assentos") / pl.col("_total_assentos")).alias("_cdf")
        )
    )

    q1 = (
        ordenado
        .filter(pl.col("_cdf") >= 0.25)
        .group_by(chaves)
        .agg(pl.col("tarifa").min().alias(f"{prefixo}_q1"))
    )

    mediana = (
        ordenado
        .filter(pl.col("_cdf") >= 0.50)
        .group_by(chaves)
        .agg(pl.col("tarifa").min().alias(f"{prefixo}_mediana"))
    )

    q3 = (
        ordenado
        .filter(pl.col("_cdf") >= 0.75)
        .group_by(chaves)
        .agg(pl.col("tarifa").min().alias(f"{prefixo}_q3"))
    )

    resultado = (
        momentos
        .join(q1, on=chaves, how="left")
        .join(mediana, on=chaves, how="left")
        .join(q3, on=chaves, how="left")
        .with_columns(
            (
                pl.col(f"{prefixo}_q3")
                - pl.col(f"{prefixo}_q1")
            ).alias(f"{prefixo}_iqr")
        )
        .with_columns([
            (
                pl.col(f"{prefixo}_desvio")
                / pl.col(f"{prefixo}_media")
            ).alias(f"{prefixo}_cv"),
            (
                pl.col(f"{prefixo}_iqr")
                / pl.col(f"{prefixo}_mediana")
            ).alias(f"{prefixo}_iqr_rel"),
        ])
        .with_columns(
            (
                1
                - (
                    -(
                        pl.col(f"{prefixo}_cv").fill_null(0).clip(lower_bound=0)
                        + pl.col(f"{prefixo}_iqr_rel").fill_null(0).clip(lower_bound=0)
                    )
                ).exp()
            )
            .clip(0, 1)
            .alias(f"{prefixo}_score_volatilidade")
        )
    )

    return resultado


## 5. Preparação da Silver histórica de voos

O target de cancelamento é binário:

- `1`: voo cancelado;
- `0`: voo realizado.

O target de atraso também é binário:

- `1`: voo atrasado;
- `0`: voo sem atraso.

Para o modelo de atraso serão utilizados apenas os voos realizados.


In [ ]:
# ============================================================
# 5.1 LEITURA E TRATAMENTO DO HISTÓRICO DE VOOS
# ============================================================

df_historico = (
    pl.scan_parquet(PATH_VOOS)
    .select([
        "municipio_origem",
        "municipio_destino",
        "nome_empresa",
        "partida_prevista",
        "partida_real",
        "situacao_voo",
        "desc_partida",
    ])
    .filter(
        pl.col("situacao_voo").is_in(["REALIZADO", "CANCELADO"])
    )
    .filter(
        pl.col("municipio_origem").is_not_null()
        & pl.col("municipio_destino").is_not_null()
        & pl.col("nome_empresa").is_not_null()
        & pl.col("partida_prevista").is_not_null()
    )
    .with_columns([
        pl.col("partida_prevista").dt.date().alias("data_historica"),

        pl.when(pl.col("situacao_voo") == "CANCELADO")
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("cancelado"),

        pl.when(
            pl.col("desc_partida")
            .fill_null("")
            .str.to_lowercase()
            == "atrasado"
        )
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("atrasado"),
    ])
    .with_columns([
        pl.col("data_historica").dt.year().alias("ano"),
        pl.col("data_historica").dt.month().alias("n_mes"),
        pl.col("data_historica").dt.day().alias("n_dia"),
        pl.col("data_historica").dt.weekday().alias("n_dia_semana"),

        pl.concat_str(
            ["municipio_origem", "municipio_destino"],
            separator=" -> "
        ).alias("rota"),

        pl.concat_str(
            ["nome_empresa", "municipio_origem", "municipio_destino"],
            separator=" | "
        ).alias("empresa_rota"),
    ])
    .with_columns(
        (
            pl.col("ano") * 12 + pl.col("n_mes")
        ).cast(pl.Int32).alias("indice_tempo")
    )
    .select([
        "municipio_origem",
        "municipio_destino",
        "nome_empresa",
        "data_historica",
        "cancelado",
        "atrasado",
        "ano",
        "n_mes",
        "n_dia",
        "n_dia_semana",
        "indice_tempo",
        "rota",
        "empresa_rota",
    ])
    .collect()
)

print(df_historico.shape)
df_historico.head()


In [ ]:
# ============================================================
# 5.2 CONTROLES DE QUALIDADE
# ============================================================

print(
    df_historico.select(
        pl.col("data_historica").min().alias("data_min"),
        pl.col("data_historica").max().alias("data_max"),
    )
)

print(
    df_historico
    .group_by("cancelado")
    .agg(pl.len().alias("qtd"))
    .with_columns(
        (pl.col("qtd") / pl.col("qtd").sum()).alias("percentual")
    )
    .sort("cancelado")
)

print(
    df_historico
    .filter(pl.col("cancelado") == 0)
    .group_by("atrasado")
    .agg(pl.len().alias("qtd"))
    .with_columns(
        (pl.col("qtd") / pl.col("qtd").sum()).alias("percentual")
    )
    .sort("atrasado")
)


## 6. Features dos modelos operacionais

Foram incluídas interações explícitas de rota e companhia:

- companhia;
- município de origem;
- município de destino;
- rota origem → destino;
- companhia + rota;
- mês;
- dia do mês;
- dia da semana;
- tendência temporal mensal.

Isso permite que os scores variem por **companhia, rota e data**.


In [ ]:
# ============================================================
# 6. CONFIGURAÇÃO DAS FEATURES
# ============================================================

FEATURES_CATEGORICAS = [
    "nome_empresa",
    "municipio_origem",
    "municipio_destino",
    "rota",
    "empresa_rota",
    "n_mes",
    "n_dia",
    "n_dia_semana",
]

FEATURES_NUMERICAS = [
    "indice_tempo",
]

FEATURES_MODELO = FEATURES_CATEGORICAS + FEATURES_NUMERICAS


def construir_pipeline_classificacao():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
                FEATURES_CATEGORICAS,
            ),
            (
                "num",
                StandardScaler(),
                FEATURES_NUMERICAS,
            ),
        ],
        remainder="drop",
    )

    # Como o objetivo é usar predict_proba como score,
    # não aplicamos class_weight='balanced' nesta versão.
    # O desbalanceamento será acompanhado por PR-AUC e Brier Score.
    modelo = LogisticRegression(
        solver="saga",
        penalty="l2",
        max_iter=500,
        tol=1e-3,
        random_state=RANDOM_STATE,
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("modelo", modelo),
    ])


## 7. Separação temporal

A validação respeita o tempo:

- treino: até 31/12/2024;
- validação: 01/01/2025 a 31/12/2025;
- teste: 01/01/2026 em diante.

Nenhum dado futuro é usado para treinar o modelo que avalia um período anterior.


In [ ]:
# ============================================================
# 7. SPLIT TEMPORAL
# ============================================================

df_train = df_historico.filter(
    pl.col("data_historica") < DATA_INICIO_VALIDACAO
)

df_valid = df_historico.filter(
    (pl.col("data_historica") >= DATA_INICIO_VALIDACAO)
    & (pl.col("data_historica") < DATA_INICIO_TESTE)
)

df_test = df_historico.filter(
    pl.col("data_historica") >= DATA_INICIO_TESTE
)

for nome, df in [
    ("Treino", df_train),
    ("Validação", df_valid),
    ("Teste", df_test),
]:
    print(
        nome,
        df.shape,
        "cancelamento:",
        round(df.select(pl.col("cancelado").mean()).item(), 6),
    )


# 8. Modelo 1: risco de cancelamento

A regressão logística fornece diretamente uma probabilidade entre 0 e 1.

As métricas principais são:

- **ROC-AUC**: capacidade de ordenação entre maior e menor risco;
- **PR-AUC**: especialmente importante devido à baixa prevalência de cancelamentos;
- **Brier Score**: avalia a qualidade das probabilidades;
- **Log Loss**: penaliza probabilidades excessivamente confiantes e erradas.


In [ ]:
# ============================================================
# 8.1 TREINO E VALIDAÇÃO DO MODELO DE CANCELAMENTO
# ============================================================

X_train_cancel = df_train.select(FEATURES_MODELO).to_pandas()
y_train_cancel = df_train["cancelado"].to_numpy()

X_valid_cancel = df_valid.select(FEATURES_MODELO).to_pandas()
y_valid_cancel = df_valid["cancelado"].to_numpy()

modelo_cancel_validacao = construir_pipeline_classificacao()

modelo_cancel_validacao.fit(
    X_train_cancel,
    y_train_cancel,
)

metricas_cancel_valid, prob_cancel_valid = avaliar_classificador(
    modelo_cancel_validacao,
    X_valid_cancel,
    y_valid_cancel,
    nome_modelo="cancelamento_logistica",
    split="validacao",
)

metricas_cancel_valid


In [ ]:
# ============================================================
# 8.2 TESTE FORA DA AMOSTRA
# ============================================================

# Após a validação, treinamos novamente usando treino + validação
df_train_valid = pl.concat([df_train, df_valid])

X_train_valid_cancel = (
    df_train_valid
    .select(FEATURES_MODELO)
    .to_pandas()
)
y_train_valid_cancel = df_train_valid["cancelado"].to_numpy()

X_test_cancel = df_test.select(FEATURES_MODELO).to_pandas()
y_test_cancel = df_test["cancelado"].to_numpy()

modelo_cancel_teste = construir_pipeline_classificacao()

modelo_cancel_teste.fit(
    X_train_valid_cancel,
    y_train_valid_cancel,
)

metricas_cancel_test, prob_cancel_test = avaliar_classificador(
    modelo_cancel_teste,
    X_test_cancel,
    y_test_cancel,
    nome_modelo="cancelamento_logistica",
    split="teste",
)

metricas_cancel_test


In [ ]:
# ============================================================
# 8.3 MODELO FINAL DE CANCELAMENTO
# ============================================================

# Depois da avaliação, o modelo de produção é treinado
# com todo o histórico disponível.

X_all_cancel = df_historico.select(FEATURES_MODELO).to_pandas()
y_all_cancel = df_historico["cancelado"].to_numpy()

modelo_cancelamento_final = construir_pipeline_classificacao()

modelo_cancelamento_final.fit(
    X_all_cancel,
    y_all_cancel,
)

joblib.dump(
    modelo_cancelamento_final,
    fr"{pasta_modelos}/modelo_cancelamento_{hoje_str}.joblib",
)

print("Modelo final de cancelamento treinado e salvo.")


# 9. Modelo 2: risco de atraso

Cancelamentos são removidos desta modelagem, pois atraso somente é observável em voos realizados.

A variável alvo é `atrasado`.


In [ ]:
# ============================================================
# 9.1 BASE ESPECÍFICA DO MODELO DE ATRASO
# ============================================================

df_historico_atraso = df_historico.filter(
    pl.col("cancelado") == 0
)

df_train_atraso = df_historico_atraso.filter(
    pl.col("data_historica") < DATA_INICIO_VALIDACAO
)

df_valid_atraso = df_historico_atraso.filter(
    (pl.col("data_historica") >= DATA_INICIO_VALIDACAO)
    & (pl.col("data_historica") < DATA_INICIO_TESTE)
)

df_test_atraso = df_historico_atraso.filter(
    pl.col("data_historica") >= DATA_INICIO_TESTE
)

for nome, df in [
    ("Treino", df_train_atraso),
    ("Validação", df_valid_atraso),
    ("Teste", df_test_atraso),
]:
    print(
        nome,
        df.shape,
        "atraso:",
        round(df.select(pl.col("atrasado").mean()).item(), 6),
    )


In [ ]:
# ============================================================
# 9.2 TREINO E VALIDAÇÃO DO MODELO DE ATRASO
# ============================================================

X_train_atraso = df_train_atraso.select(FEATURES_MODELO).to_pandas()
y_train_atraso = df_train_atraso["atrasado"].to_numpy()

X_valid_atraso = df_valid_atraso.select(FEATURES_MODELO).to_pandas()
y_valid_atraso = df_valid_atraso["atrasado"].to_numpy()

modelo_atraso_validacao = construir_pipeline_classificacao()

modelo_atraso_validacao.fit(
    X_train_atraso,
    y_train_atraso,
)

metricas_atraso_valid, prob_atraso_valid = avaliar_classificador(
    modelo_atraso_validacao,
    X_valid_atraso,
    y_valid_atraso,
    nome_modelo="atraso_logistica",
    split="validacao",
)

metricas_atraso_valid


In [ ]:
# ============================================================
# 9.3 TESTE FORA DA AMOSTRA
# ============================================================

df_train_valid_atraso = pl.concat([
    df_train_atraso,
    df_valid_atraso,
])

X_train_valid_atraso = (
    df_train_valid_atraso
    .select(FEATURES_MODELO)
    .to_pandas()
)
y_train_valid_atraso = df_train_valid_atraso["atrasado"].to_numpy()

X_test_atraso = df_test_atraso.select(FEATURES_MODELO).to_pandas()
y_test_atraso = df_test_atraso["atrasado"].to_numpy()

modelo_atraso_teste = construir_pipeline_classificacao()

modelo_atraso_teste.fit(
    X_train_valid_atraso,
    y_train_valid_atraso,
)

metricas_atraso_test, prob_atraso_test = avaliar_classificador(
    modelo_atraso_teste,
    X_test_atraso,
    y_test_atraso,
    nome_modelo="atraso_logistica",
    split="teste",
)

metricas_atraso_test


In [ ]:
# ============================================================
# 9.4 MODELO FINAL DE ATRASO
# ============================================================

X_all_atraso = (
    df_historico_atraso
    .select(FEATURES_MODELO)
    .to_pandas()
)
y_all_atraso = df_historico_atraso["atrasado"].to_numpy()

modelo_atraso_final = construir_pipeline_classificacao()

modelo_atraso_final.fit(
    X_all_atraso,
    y_all_atraso,
)

joblib.dump(
    modelo_atraso_final,
    fr"{pasta_modelos}/modelo_atraso_{hoje_str}.joblib",
)

print("Modelo final de atraso treinado e salvo.")


# 10. Histórico explicativo de risco operacional

Os percentuais abaixo não são usados para vazar informação para validação/teste.

Eles são calculados **após a modelagem**, usando todo o histórico disponível, e serão anexados à Gold futura para explicar o score produzido pelo ML.


In [ ]:
# ============================================================
# 10.1 FUNÇÃO PARA AGREGADOS HISTÓRICOS
# ============================================================

def agregar_risco_historico(df, chaves, prefixo):
    return (
        df
        .group_by(chaves)
        .agg([
            pl.len().alias(f"{prefixo}_voos_total"),
            pl.col("cancelado").sum().alias(f"{prefixo}_voos_cancelados"),

            (pl.col("cancelado") == 0)
            .cast(pl.Int64)
            .sum()
            .alias(f"{prefixo}_voos_realizados"),

            (
                (pl.col("cancelado") == 0)
                & (pl.col("atrasado") == 1)
            )
            .cast(pl.Int64)
            .sum()
            .alias(f"{prefixo}_voos_atrasados"),
        ])
        .with_columns([
            (
                pl.col(f"{prefixo}_voos_cancelados")
                / pl.col(f"{prefixo}_voos_total")
            ).alias(f"{prefixo}_pct_cancelamento"),

            (
                pl.col(f"{prefixo}_voos_atrasados")
                / pl.col(f"{prefixo}_voos_realizados")
            ).alias(f"{prefixo}_pct_atraso"),
        ])
    )


hist_empresa = agregar_risco_historico(
    df_historico,
    ["nome_empresa"],
    "hist_empresa",
)

hist_rota = agregar_risco_historico(
    df_historico,
    ["municipio_origem", "municipio_destino"],
    "hist_rota",
)

hist_empresa_rota = agregar_risco_historico(
    df_historico,
    ["nome_empresa", "municipio_origem", "municipio_destino"],
    "hist_empresa_rota",
)

hist_data = agregar_risco_historico(
    df_historico,
    ["n_mes", "n_dia"],
    "hist_data",
)

hist_empresa.head()


# 11. Matriz de simulações futuras

A matriz contém somente combinações de companhia e rota observadas no histórico e é cruzada com todas as datas futuras até 31/12/2027.

Caso você já possua um `df_simulacoes` equivalente, esta célula pode ser substituída pela leitura da sua tabela.


In [ ]:
# ============================================================
# 11.1 ROTAS/COMPANHIAS VÁLIDAS
# ============================================================

df_rotas_empresas = (
    df_historico
    .select([
        "municipio_origem",
        "municipio_destino",
        "nome_empresa",
    ])
    .unique()
)

ls_datas = []
data_aux = DATA_INICIAL_SIMULACAO

while data_aux <= DATA_FINAL_SIMULACAO:
    ls_datas.append(data_aux)
    data_aux += timedelta(days=1)

df_datas_futuras = pl.DataFrame({
    "datas": ls_datas
})

df_simulacoes = (
    df_rotas_empresas
    .join(df_datas_futuras, how="cross")
    .with_columns([
        pl.col("datas").dt.year().alias("ano"),
        pl.col("datas").dt.month().alias("n_mes"),
        pl.col("datas").dt.day().alias("n_dia"),
        pl.col("datas").dt.weekday().alias("n_dia_semana"),

        pl.concat_str(
            ["municipio_origem", "municipio_destino"],
            separator=" -> "
        ).alias("rota"),

        pl.concat_str(
            ["nome_empresa", "municipio_origem", "municipio_destino"],
            separator=" | "
        ).alias("empresa_rota"),
    ])
    .with_columns(
        (
            pl.col("ano") * 12 + pl.col("n_mes")
        ).cast(pl.Int32).alias("indice_tempo")
    )
)

print(df_simulacoes.shape)
df_simulacoes.head()


In [ ]:
# ============================================================
# 11.2 SCORING FUTURO DE CANCELAMENTO E ATRASO
# ============================================================

score_cancelamento = prever_probabilidade_em_lotes(
    modelo_cancelamento_final,
    df_simulacoes,
    FEATURES_MODELO,
)

score_atraso = prever_probabilidade_em_lotes(
    modelo_atraso_final,
    df_simulacoes,
    FEATURES_MODELO,
)

df_simulacoes = df_simulacoes.with_columns([
    pl.Series(
        name="score_cancelamento",
        values=score_cancelamento,
        dtype=pl.Float64,
    ),
    pl.Series(
        name="score_atraso",
        values=score_atraso,
        dtype=pl.Float64,
    ),
])

df_simulacoes.select([
    "municipio_origem",
    "municipio_destino",
    "nome_empresa",
    "datas",
    "score_cancelamento",
    "score_atraso",
]).head()


In [ ]:
# ============================================================
# 11.3 JOIN DOS INDICADORES HISTÓRICOS EXPLICATIVOS
# ============================================================

df_simulacoes = (
    df_simulacoes
    .join(
        hist_empresa,
        on=["nome_empresa"],
        how="left",
    )
    .join(
        hist_rota,
        on=["municipio_origem", "municipio_destino"],
        how="left",
    )
    .join(
        hist_empresa_rota,
        on=["nome_empresa", "municipio_origem", "municipio_destino"],
        how="left",
    )
    .join(
        hist_data,
        on=["n_mes", "n_dia"],
        how="left",
    )
)

df_simulacoes.head()


# 12. Histórico de tarifas

A tabela de tarifas é usada em duas frentes:

1. estatística histórica ponderada por assentos;
2. regressão temporal para estimar uma referência de preço futuro.

A modelagem de preço trabalha por mês porque a Silver de tarifas possui granularidade temporal `ano + n_mes`.


In [ ]:
# ============================================================
# 12.1 LEITURA DA SILVER DE TARIFAS
# ============================================================

if len(glob.glob(PATH_TARIFAS)) == 0:
    raise FileNotFoundError(
        "Nenhum parquet de tarifas foi encontrado em: "
        f"{PATH_TARIFAS}. Ajuste PATH_TARIFAS na célula de configuração."
    )

df_tarifas = (
    pl.scan_parquet(PATH_TARIFAS)
    .select([
        "ano",
        "n_mes",
        "tarifa",
        "assentos",
        "nome_empresa",
        "municipio_origem",
        "municipio_destino",
    ])
    .filter(
        pl.col("ano").is_not_null()
        & pl.col("n_mes").is_not_null()
        & pl.col("tarifa").is_not_null()
        & pl.col("assentos").is_not_null()
        & pl.col("nome_empresa").is_not_null()
        & pl.col("municipio_origem").is_not_null()
        & pl.col("municipio_destino").is_not_null()
        & (pl.col("tarifa") > 0)
        & (pl.col("assentos") > 0)
    )
    .with_columns([
        pl.concat_str(
            ["municipio_origem", "municipio_destino"],
            separator=" -> "
        ).alias("rota"),

        (
            pl.col("ano") * 12 + pl.col("n_mes")
        ).cast(pl.Int32).alias("indice_tempo"),
    ])
    .collect()
)

print(df_tarifas.shape)
df_tarifas.head()


In [ ]:
# ============================================================
# 12.2 ESTATÍSTICAS DE PREÇO: EMPRESA + ROTA + MÊS
# ============================================================

CHAVES_PRECO_EMPRESA_ROTA = [
    "nome_empresa",
    "municipio_origem",
    "municipio_destino",
    "n_mes",
]

CHAVES_PRECO_ROTA = [
    "municipio_origem",
    "municipio_destino",
    "n_mes",
]

preco_stats_empresa_rota = estatisticas_preco_ponderadas(
    df_tarifas,
    CHAVES_PRECO_EMPRESA_ROTA,
    prefixo="preco_er",
)

# Fallback quando não existir histórico de tarifa para
# uma empresa específica naquele trecho/mês.
preco_stats_rota = estatisticas_preco_ponderadas(
    df_tarifas,
    CHAVES_PRECO_ROTA,
    prefixo="preco_rota",
)

preco_stats_empresa_rota.head()


# 13. Regressão temporal de preço

Para evitar treinar um modelo independente para cada rota, é utilizado um modelo global com:

- companhia;
- rota;
- mês do ano;
- índice temporal mensal.

O target é `log(1 + tarifa média ponderada do mês)`. A transformação logarítmica reduz a influência de tarifas extremas e garante previsões positivas após a transformação inversa.


In [ ]:
# ============================================================
# 13.1 AGREGAÇÃO MENSAL PARA A REGRESSÃO
# ============================================================

df_preco_mensal = (
    df_tarifas
    .with_columns(
        (pl.col("tarifa") * pl.col("assentos")).alias("_wx")
    )
    .group_by([
        "ano",
        "n_mes",
        "nome_empresa",
        "municipio_origem",
        "municipio_destino",
        "rota",
        "indice_tempo",
    ])
    .agg([
        pl.col("assentos").sum().alias("assentos_mes"),
        pl.col("_wx").sum().alias("_sum_wx"),
    ])
    .with_columns(
        (
            pl.col("_sum_wx")
            / pl.col("assentos_mes")
        ).alias("preco_medio_mes")
    )
    .drop("_sum_wx")
    .with_columns(
        pl.date(
            pl.col("ano"),
            pl.col("n_mes"),
            pl.lit(1),
        ).alias("data_mes")
    )
)

print(df_preco_mensal.shape)
df_preco_mensal.head()


In [ ]:
# ============================================================
# 13.2 SPLIT TEMPORAL DA BASE DE PREÇOS
# ============================================================

meses_disponiveis = (
    df_preco_mensal
    .select("data_mes")
    .unique()
    .sort("data_mes")
    .to_series()
    .to_list()
)

if len(meses_disponiveis) < 12:
    raise ValueError(
        "Histórico de tarifas insuficiente para uma validação temporal adequada."
    )

# Divisão 70% / 15% / 15% por meses, garantindo ordem temporal.
idx_valid = int(len(meses_disponiveis) * 0.70)
idx_test = int(len(meses_disponiveis) * 0.85)

corte_valid_preco = meses_disponiveis[idx_valid]
corte_test_preco = meses_disponiveis[idx_test]

df_preco_train = df_preco_mensal.filter(
    pl.col("data_mes") < corte_valid_preco
)

df_preco_valid = df_preco_mensal.filter(
    (pl.col("data_mes") >= corte_valid_preco)
    & (pl.col("data_mes") < corte_test_preco)
)

df_preco_test = df_preco_mensal.filter(
    pl.col("data_mes") >= corte_test_preco
)

print("Corte validação:", corte_valid_preco)
print("Corte teste:", corte_test_preco)

print("Treino:", df_preco_train.shape)
print("Validação:", df_preco_valid.shape)
print("Teste:", df_preco_test.shape)


In [ ]:
# ============================================================
# 13.3 PIPELINE DA REGRESSÃO DE PREÇO
# ============================================================

FEATURES_PRECO_CATEGORICAS = [
    "nome_empresa",
    "rota",
    "n_mes",
]

FEATURES_PRECO_NUMERICAS = [
    "indice_tempo",
]

FEATURES_PRECO = (
    FEATURES_PRECO_CATEGORICAS
    + FEATURES_PRECO_NUMERICAS
)


def construir_pipeline_preco():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
                FEATURES_PRECO_CATEGORICAS,
            ),
            (
                "num",
                StandardScaler(),
                FEATURES_PRECO_NUMERICAS,
            ),
        ],
        remainder="drop",
    )

    regressao = Ridge(
        alpha=10.0,
        solver="lsqr",
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("modelo", regressao),
    ])


In [ ]:
# ============================================================
# 13.4 VALIDAÇÃO DO MODELO DE PREÇO
# ============================================================

X_preco_train = (
    df_preco_train
    .select(FEATURES_PRECO)
    .to_pandas()
)
y_preco_train = np.log1p(
    df_preco_train["preco_medio_mes"].to_numpy()
)

X_preco_valid = (
    df_preco_valid
    .select(FEATURES_PRECO)
    .to_pandas()
)
y_preco_valid_real = (
    df_preco_valid["preco_medio_mes"].to_numpy()
)

modelo_preco_validacao = construir_pipeline_preco()

modelo_preco_validacao.fit(
    X_preco_train,
    y_preco_train,
)

pred_preco_valid = np.maximum(
    np.expm1(modelo_preco_validacao.predict(X_preco_valid)),
    0,
)

metricas_preco_valid = avaliar_regressor(
    y_preco_valid_real,
    pred_preco_valid,
    nome_modelo="preco_ridge_temporal",
    split="validacao",
)

metricas_preco_valid


In [ ]:
# ============================================================
# 13.5 TESTE FORA DA AMOSTRA
# ============================================================

df_preco_train_valid = pl.concat([
    df_preco_train,
    df_preco_valid,
])

X_preco_train_valid = (
    df_preco_train_valid
    .select(FEATURES_PRECO)
    .to_pandas()
)
y_preco_train_valid = np.log1p(
    df_preco_train_valid["preco_medio_mes"].to_numpy()
)

X_preco_test = (
    df_preco_test
    .select(FEATURES_PRECO)
    .to_pandas()
)
y_preco_test_real = (
    df_preco_test["preco_medio_mes"].to_numpy()
)

modelo_preco_teste = construir_pipeline_preco()

modelo_preco_teste.fit(
    X_preco_train_valid,
    y_preco_train_valid,
)

pred_preco_test = np.maximum(
    np.expm1(modelo_preco_teste.predict(X_preco_test)),
    0,
)

metricas_preco_test = avaliar_regressor(
    y_preco_test_real,
    pred_preco_test,
    nome_modelo="preco_ridge_temporal",
    split="teste",
)

metricas_preco_test


In [ ]:
# ============================================================
# 13.6 MODELO FINAL DE PREÇO
# ============================================================

X_preco_all = (
    df_preco_mensal
    .select(FEATURES_PRECO)
    .to_pandas()
)

y_preco_all = np.log1p(
    df_preco_mensal["preco_medio_mes"].to_numpy()
)

modelo_preco_final = construir_pipeline_preco()

modelo_preco_final.fit(
    X_preco_all,
    y_preco_all,
)

joblib.dump(
    modelo_preco_final,
    fr"{pasta_modelos}/modelo_preco_{hoje_str}.joblib",
)

print("Modelo final de preço treinado e salvo.")


In [ ]:
# ============================================================
# 13.7 PREVISÃO DE PREÇO PARA TODAS AS SIMULAÇÕES FUTURAS
# ============================================================

preco_estimado = prever_regressao_em_lotes(
    modelo_preco_final,
    df_simulacoes,
    FEATURES_PRECO,
)

df_simulacoes = df_simulacoes.with_columns(
    pl.Series(
        name="preco_estimado_modelo",
        values=preco_estimado,
        dtype=pl.Float64,
    )
)

df_simulacoes.select([
    "municipio_origem",
    "municipio_destino",
    "nome_empresa",
    "datas",
    "preco_estimado_modelo",
]).head()


In [ ]:
# ============================================================
# 14. JOIN DAS REFERÊNCIAS HISTÓRICAS DE PREÇO
# ============================================================

df_gold = (
    df_simulacoes
    .join(
        preco_stats_empresa_rota,
        on=CHAVES_PRECO_EMPRESA_ROTA,
        how="left",
    )
    .join(
        preco_stats_rota,
        on=CHAVES_PRECO_ROTA,
        how="left",
    )
)

# Prioridade:
# 1. companhia + rota + mês
# 2. rota + mês
# 3. sem referência histórica

df_gold = (
    df_gold
    .with_columns(
        pl.when(pl.col("preco_er_media").is_not_null())
        .then(pl.lit("empresa_rota_mes"))
        .when(pl.col("preco_rota_media").is_not_null())
        .then(pl.lit("rota_mes"))
        .otherwise(pl.lit("sem_historico"))
        .alias("nivel_referencia_preco")
    )
    .with_columns([
        pl.coalesce([
            pl.col("preco_er_media"),
            pl.col("preco_rota_media"),
        ]).alias("preco_historico_media"),

        pl.coalesce([
            pl.col("preco_er_mediana"),
            pl.col("preco_rota_mediana"),
        ]).alias("preco_historico_mediana"),

        pl.coalesce([
            pl.col("preco_er_desvio"),
            pl.col("preco_rota_desvio"),
        ]).alias("preco_historico_desvio"),

        pl.coalesce([
            pl.col("preco_er_q1"),
            pl.col("preco_rota_q1"),
        ]).alias("preco_historico_q1"),

        pl.coalesce([
            pl.col("preco_er_q3"),
            pl.col("preco_rota_q3"),
        ]).alias("preco_historico_q3"),

        pl.coalesce([
            pl.col("preco_er_iqr"),
            pl.col("preco_rota_iqr"),
        ]).alias("preco_historico_iqr"),

        pl.coalesce([
            pl.col("preco_er_score_volatilidade"),
            pl.col("preco_rota_score_volatilidade"),
        ]).alias("score_volatilidade_preco"),
    ])
)


# 15. Scores consolidados

Nesta fase ainda não existe `preco_atual`.

Por isso:

- `score_risco_operacional` combina cancelamento e atraso;
- `score_volatilidade_preco` mede a instabilidade histórica da tarifa;
- `score_risco_base` combina os três pilares históricos.

Quando a API for adicionada, o preço observado poderá ser comparado com:

- `preco_historico_mediana`;
- `preco_historico_q1`;
- `preco_historico_q3`;
- `preco_historico_iqr`;
- `preco_historico_desvio`;
- `preco_estimado_modelo`.

A partir disso será criado o futuro `score_preco_atual` e o `score_risco_geral`.


In [ ]:
# ============================================================
# 15.1 CÁLCULO DOS SCORES CONSOLIDADOS
# ============================================================

df_gold = (
    df_gold
    .with_columns(
        (
            (
                pl.col("score_cancelamento")
                + pl.col("score_atraso")
            )
            / 2
        ).alias("score_risco_operacional")
    )
    .with_columns(
        pl.when(
            pl.col("score_volatilidade_preco").is_not_null()
        )
        .then(
            (
                pl.col("score_cancelamento")
                + pl.col("score_atraso")
                + pl.col("score_volatilidade_preco")
            )
            / 3
        )
        .otherwise(
            pl.col("score_risco_operacional")
        )
        .alias("score_risco_base")
    )
    .with_columns([
        pl.lit(hoje).cast(pl.Date).alias("data_processamento"),
        pl.lit(hoje_str).alias("versao_modelo"),
    ])
)

df_gold.select([
    "municipio_origem",
    "municipio_destino",
    "nome_empresa",
    "datas",
    "score_cancelamento",
    "score_atraso",
    "score_volatilidade_preco",
    "score_risco_operacional",
    "score_risco_base",
    "preco_estimado_modelo",
    "preco_historico_mediana",
    "preco_historico_q1",
    "preco_historico_q3",
]).head()


# 16. Seleção final da Gold/SPEC

A tabela final mantém:

1. chaves de consulta;
2. scores de ML;
3. indicadores históricos explicativos;
4. referências estatísticas de preço;
5. versão e data de processamento.

Essa estrutura foi desenhada para permitir posteriormente um `JOIN` simples com o resultado da SerpApi.


In [ ]:
# ============================================================
# 16.1 GOLD FINAL
# ============================================================

COLUNAS_GOLD = [
    # Chaves
    "datas",
    "municipio_origem",
    "municipio_destino",
    "nome_empresa",

    # Features temporais úteis no consumo
    "ano",
    "n_mes",
    "n_dia",
    "n_dia_semana",

    # Scores ML
    "score_cancelamento",
    "score_atraso",
    "score_risco_operacional",

    # Histórico de empresa
    "hist_empresa_voos_total",
    "hist_empresa_voos_cancelados",
    "hist_empresa_voos_realizados",
    "hist_empresa_voos_atrasados",
    "hist_empresa_pct_cancelamento",
    "hist_empresa_pct_atraso",

    # Histórico da rota
    "hist_rota_voos_total",
    "hist_rota_voos_cancelados",
    "hist_rota_voos_realizados",
    "hist_rota_voos_atrasados",
    "hist_rota_pct_cancelamento",
    "hist_rota_pct_atraso",

    # Histórico companhia + rota
    "hist_empresa_rota_voos_total",
    "hist_empresa_rota_voos_cancelados",
    "hist_empresa_rota_voos_realizados",
    "hist_empresa_rota_voos_atrasados",
    "hist_empresa_rota_pct_cancelamento",
    "hist_empresa_rota_pct_atraso",

    # Histórico da combinação mês/dia
    "hist_data_voos_total",
    "hist_data_voos_cancelados",
    "hist_data_voos_realizados",
    "hist_data_voos_atrasados",
    "hist_data_pct_cancelamento",
    "hist_data_pct_atraso",

    # Preço
    "nivel_referencia_preco",
    "preco_estimado_modelo",
    "preco_historico_media",
    "preco_historico_mediana",
    "preco_historico_desvio",
    "preco_historico_q1",
    "preco_historico_q3",
    "preco_historico_iqr",
    "score_volatilidade_preco",

    # Score consolidado sem preço atual
    "score_risco_base",

    # Auditoria
    "data_processamento",
    "versao_modelo",
]

df_gold_final = (
    df_gold
    .select(COLUNAS_GOLD)
    .rename({"datas": "data_voo"})
    .sort([
        "data_voo",
        "municipio_origem",
        "municipio_destino",
        "nome_empresa",
    ])
)

print(df_gold_final.shape)
df_gold_final.head()


In [ ]:
# ============================================================
# 17. MÉTRICAS DOS MODELOS
# ============================================================

metricas_classificacao = [
    metricas_cancel_valid,
    metricas_cancel_test,
    metricas_atraso_valid,
    metricas_atraso_test,
]

df_metricas_classificacao = pl.DataFrame(
    metricas_classificacao
)

metricas_regressao = [
    metricas_preco_valid,
    metricas_preco_test,
]

df_metricas_regressao = pl.DataFrame(
    metricas_regressao
)

print("CLASSIFICAÇÃO")
print(df_metricas_classificacao)

print("\nPREÇO")
print(df_metricas_regressao)


In [ ]:
# ============================================================
# 18. MATERIALIZAÇÃO DAS TABELAS SPEC
# ============================================================

pasta_gold_risco = fr"{pasta_spec}/spec_modelos_risco"
pasta_metricas = fr"{pasta_spec}/spec_metricas_modelos"

os.makedirs(pasta_gold_risco, exist_ok=True)
os.makedirs(pasta_metricas, exist_ok=True)

arquivo_gold = (
    fr"{pasta_gold_risco}/"
    fr"spec_modelos_risco_{hoje_str}.parquet"
)

arquivo_metricas_class = (
    fr"{pasta_metricas}/"
    fr"metricas_classificacao_{hoje_str}.parquet"
)

arquivo_metricas_preco = (
    fr"{pasta_metricas}/"
    fr"metricas_preco_{hoje_str}.parquet"
)

df_gold_final.write_parquet(
    arquivo_gold,
    compression="zstd",
)

df_metricas_classificacao.write_parquet(
    arquivo_metricas_class,
    compression="zstd",
)

df_metricas_regressao.write_parquet(
    arquivo_metricas_preco,
    compression="zstd",
)

print("Gold materializada em:")
print(arquivo_gold)

print("\nMétricas materializadas em:")
print(arquivo_metricas_class)
print(arquivo_metricas_preco)


In [ ]:
# ============================================================
# 19. VALIDAÇÕES FINAIS DA GOLD
# ============================================================

print("Shape:", df_gold_final.shape)

print(
    df_gold_final.select([
        pl.col("data_voo").min().alias("data_min"),
        pl.col("data_voo").max().alias("data_max"),
        pl.col("score_cancelamento").min().alias("cancel_min"),
        pl.col("score_cancelamento").max().alias("cancel_max"),
        pl.col("score_atraso").min().alias("atraso_min"),
        pl.col("score_atraso").max().alias("atraso_max"),
        pl.col("score_risco_base").min().alias("risco_min"),
        pl.col("score_risco_base").max().alias("risco_max"),
    ])
)

duplicidades = (
    df_gold_final
    .group_by([
        "data_voo",
        "municipio_origem",
        "municipio_destino",
        "nome_empresa",
    ])
    .len()
    .filter(pl.col("len") > 1)
)

print("Combinações duplicadas:", duplicidades.height)

nulos_scores = df_gold_final.select([
    pl.col("score_cancelamento").null_count().alias("null_cancelamento"),
    pl.col("score_atraso").null_count().alias("null_atraso"),
    pl.col("preco_estimado_modelo").null_count().alias("null_preco_estimado"),
])

print(nulos_scores)
